Code for Assingment: Study group 3






# Assignment 2: Automated Prompt Optimisation (APO) with OPRO on BBH Navigate

This notebook builds a complete Automated Prompt Optimisation (APO) pipeline using the OPRO strategy.

## Objective
We optimise a system prompt for the BIG-Bench Hard (BBH) `navigate` task, where the model must determine whether a sequence of navigation instructions returns to the starting point.

## Required assignment components implemented
- Load `lukaemon/bbh` with config `navigate`
- Use rows `[0:20]` as train and `[20:40]` as test
- Establish a naive baseline on both train and test
- Build a meta-prompt containing prior prompts and scores
- Run an iterative OPRO loop with an optimiser LLM
- Automatically extract and evaluate new prompts
- Track the optimisation trajectory with logs
- Evaluate the best final prompt on held-out test data only at the end

## Models
- Target LLM: `gemini/gemma-3-4b-it`
- Optimiser LLM: `gemini/gemma-3-27b-it`

## Notes
- Rate limiting is handled with `asyncio.sleep(...)`
- Logs are saved to disk
- Final results are saved for the writeup


## Setup

Install dependencies and import all required libraries. `nest_asyncio` is applied so that `asyncio` event loops work correctly inside Jupyter.

In [9]:
# If running in Colab / fresh environment, uncomment:
# !pip install litellm datasets pandas nest_asyncio wandb

%matplotlib inline

import wandb
import matplotlib.pyplot as plt
import os
import re
import json
import time
import math
import asyncio
from datetime import datetime
from pathlib import Path

import pandas as pd
from datasets import load_dataset
from litellm import acompletion

# Useful in notebooks if an event loop is already running
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass



## Configuration

Central configuration block. Changing `TARGET_MODEL` or `OPTIMISER_MODEL` here affects the entire pipeline. `MAX_OPRO_ITERS` caps the number of optimisation rounds and `TARGET_SCORE_STOP` triggers early exit if train accuracy reaches that threshold. The sleep constants keep API calls within the free-tier RPM limit.

In [11]:
# =========================
# CONFIG
# =========================

TARGET_MODEL = "gemini/gemma-3-4b-it"
OPTIMISER_MODEL = "gemini/gemma-3-27b-it"

# Optional Groq alternative if needed:
# TARGET_MODEL = "groq/gemma2-9b-it"
# OPTIMISER_MODEL = "groq/llama-3.3-70b-versatile"

TARGET_TEMPERATURE = 0.0
OPTIMISER_TEMPERATURE = 0.7

MAX_OPRO_ITERS = 10
TARGET_SCORE_STOP = 0.95  # stop early if train accuracy reaches this

# Rate limiting:
# Free Gemini tiers often have RPM constraints; we stay conservative.
SLEEP_BETWEEN_TARGET_CALLS = 2.2
SLEEP_BETWEEN_OPTIMISER_CALLS = 2.5

# Logging / outputs
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = Path(f"apo_run_{RUN_ID}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LOG_FILE = OUTPUT_DIR / "opro_log.txt"
TRAJECTORY_JSON = OUTPUT_DIR / "trajectory.json"
RESULTS_JSON = OUTPUT_DIR / "final_results.json"

print("Output directory:", OUTPUT_DIR.resolve())
print("Target model:", TARGET_MODEL)
print("Optimiser model:", OPTIMISER_MODEL)


Output directory: /content/apo_run_20260315_204307
Target model: gemini/gemma-3-4b-it
Optimiser model: gemini/gemma-3-27b-it


## API Key

The Gemini API key is entered interactively via `getpass` so it is never stored in plain text in the notebook.

In [12]:
import os
from getpass import getpass

# Paste your Gemini key when prompted. It will be hidden as you type.
os.environ["GEMINI_API_KEY"] = getpass("Enter your Gemini API key:").strip()

assert os.getenv("GEMINI_API_KEY"), "No key was entered."

print("GEMINI_API_KEY set successfully.")


Enter your Gemini API key:··········
GEMINI_API_KEY set successfully.


In [13]:
os.environ["WANDB_API_KEY"] = getpass("Enter your W&B API key:").strip()
wandb.login(key=os.environ["WANDB_API_KEY"])
print("W&B login successful.")


Enter your W&B API key:··········


wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


W&B login successful.


## API Key Validation

Checks that the correct environment variable is set depending on whether Gemini or Groq models are selected. This fails fast before any expensive API calls are made.

In [14]:
# Gemini:
if TARGET_MODEL.startswith("gemini/") or OPTIMISER_MODEL.startswith("gemini/"):
    assert os.getenv("GEMINI_API_KEY"), "Missing GEMINI_API_KEY environment variable."

# Groq:
if TARGET_MODEL.startswith("groq/") or OPTIMISER_MODEL.startswith("groq/"):
    assert os.getenv("GROQ_API_KEY"), "Missing GROQ_API_KEY environment variable."

print("API key check passed.")


API key check passed.


In [15]:
wandb.init(
    project="apo-navigate",
    name=RUN_ID,
    config={
        "target_model": TARGET_MODEL,
        "optimiser_model": OPTIMISER_MODEL,
        "max_opro_iters": MAX_OPRO_ITERS,
        "target_score_stop": TARGET_SCORE_STOP,
        "train_size": 20,
        "test_size": 20,
        "sleep_target": SLEEP_BETWEEN_TARGET_CALLS,
        "sleep_optimiser": SLEEP_BETWEEN_OPTIMISER_CALLS,
    }
)
from IPython.display import display, HTML
display(HTML(f'<b>W&amp;B Run started:</b> <a href="{wandb.run.url}" target="_blank">{wandb.run.url}</a>'))


[2026-03-15 20:43:34] W&B run initialised: https://wandb.ai/praneeth-lakkim-london-business-school/apo-navigate/runs/q4rejfst


## Logging

All pipeline events are written to a timestamped log file inside the run output directory and optionally printed to the notebook. This produces the required log deliverable automatically.

In [16]:
# =========================
# LOGGING HELPERS
# =========================

def log(msg: str, also_print: bool = True):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{timestamp}] {msg}"
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(line + "\n")
    if also_print:
        print(line)

log("=== Starting APO / OPRO run ===")
log(f"Run ID: {RUN_ID}")
log(f"Target model: {TARGET_MODEL}")
log(f"Optimiser model: {OPTIMISER_MODEL}")


[2026-03-15 20:43:34] === Starting APO / OPRO run ===
[2026-03-15 20:43:34] Run ID: 20260315_204307
[2026-03-15 20:43:34] Target model: gemini/gemma-3-4b-it
[2026-03-15 20:43:34] Optimiser model: gemini/gemma-3-27b-it


## Data Loading

Loads the `navigate` configuration of the `lukaemon/bbh` dataset from Hugging Face. The dataset is split exactly as required by the assignment:

- **Training set**: first 20 records `[0:20]`
- **Test set**: next 20 records `[20:40]`

The test set is kept entirely hidden until the final evaluation step.

In [17]:
# =========================
# DATA LOADING
# =========================

dataset = load_dataset("lukaemon/bbh", "navigate")
data = dataset["test"]

df_train = pd.DataFrame({
    "question": data["input"][:20],
    "answer": data["target"][:20]
})

df_test = pd.DataFrame({
    "question": data["input"][20:40],
    "answer": data["target"][20:40]
})

print("Train size:", len(df_train))
print("Test size:", len(df_test))
display(df_train.head(3))
display(df_test.head(3))

log("Loaded lukaemon/bbh with config navigate")
log(f"Train size: {len(df_train)}")
log(f"Test size: {len(df_test)}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

navigate/test-00000-of-00001.parquet:   0%|          | 0.00/9.54k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/250 [00:00<?, ? examples/s]

Train size: 20
Test size: 20


,question,answer
0,"If you follow these instructions, do you retur...",No
1,"If you follow these instructions, do you retur...",No
2,"If you follow these instructions, do you retur...",No


,question,answer
0,"If you follow these instructions, do you retur...",No
1,"If you follow these instructions, do you retur...",Yes
2,"If you follow these instructions, do you retur...",No


[2026-03-15 20:43:39] Loaded lukaemon/bbh with config navigate
[2026-03-15 20:43:39] Train size: 20
[2026-03-15 20:43:39] Test size: 20


## Prompt Templates

Two starting prompts are defined:

- **`BASELINE_PROMPT`** — a deliberately naive prompt used to establish the pre-optimisation accuracy.
- **`SEED_PROMPT`** — a more structured starting point that introduces explicit coordinate tracking at `(0, 0)`. Both are fed into the OPRO loop as the initial prompt pool.

In [18]:
# =========================
# PROMPT TEMPLATES
# =========================

BASELINE_PROMPT = """Solve the navigation problem.
Determine whether the final position is the same as the starting position.

Reason carefully about directions and movement.
At the end, answer with exactly one word: Yes or No."""

SEED_PROMPT = """You are solving a spatial navigation reasoning task.

Instructions:
1. Start at coordinate (0, 0).
2. Convert each movement instruction into an update of x/y position.
3. Track the net displacement carefully.
4. At the end, check whether the final coordinate is exactly (0, 0).
5. Output only one final answer: Yes or No.

Be careful with:
- left vs right
- forward vs backward
- repeated moves
- multi-step accumulation
"""


## Answer Parsing

Because the target model may produce verbose output (chain-of-thought, filler text), `normalize_answer()` applies a hierarchy of regex patterns to extract the final `yes` or `no` reliably. `extract_prompt_from_optimiser_response()` parses the optimiser's proposed prompt from `<PROMPT>` tags or fenced code blocks.

In [19]:
# =========================
# PARSING HELPERS
# =========================

def normalize_answer(text: str) -> str:
    if text is None:
        return "unknown"

    t = text.strip().lower()

    final_patterns = [
        r"final answer\s*[:\-]\s*(yes|no)\b",
        r"answer\s*[:\-]\s*(yes|no)\b",
        r"^\s*(yes|no)\s*$",
        r"\b(yes|no)\b"
    ]

    for pat in final_patterns:
        m = re.search(pat, t, flags=re.IGNORECASE)
        if m:
            return m.group(1).lower()

    if "yes" in t and "no" not in t:
        return "yes"
    if "no" in t and "yes" not in t:
        return "no"

    return "unknown"


def extract_between_tags(text: str, start_tag: str, end_tag: str):
    pattern = re.escape(start_tag) + r"(.*?)" + re.escape(end_tag)
    match = re.search(pattern, text, flags=re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(1).strip()
    return None


def extract_prompt_from_optimiser_response(text: str) -> str:
    tagged = extract_between_tags(text, "<PROMPT>", "</PROMPT>")
    if tagged:
        return tagged.strip()

    fenced = re.findall(r"```(?:text|prompt)?\n(.*?)```", text, flags=re.DOTALL | re.IGNORECASE)
    if fenced:
        return fenced[0].strip()

    return text.strip()


## LLM Call Helper

`chat_completion` is a thin async wrapper around `litellm.acompletion`. It handles errors gracefully and accepts an optional `sleep_after` delay used throughout the pipeline to stay within the Gemini free-tier rate limit of ~30 RPM.

In [20]:
# =========================
# LLM CALL HELPER
# =========================

async def chat_completion(
    model: str,
    system_prompt: str,
    user_prompt: str,
    temperature: float = 0.0,
    max_tokens: int = 512,
    sleep_after: float = 0.0,
) -> str:
    try:
        response = await acompletion(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            temperature=temperature,
            max_tokens=max_tokens,
        )
        content = response["choices"][0]["message"]["content"]
    except Exception as e:
        content = f"ERROR: {type(e).__name__}: {str(e)}"

    if sleep_after > 0:
        await asyncio.sleep(sleep_after)

    return content


## Connectivity Test

A quick sanity check that the API key is valid and the target model is reachable before committing to a full optimisation run.

In [21]:
# =========================
# CONNECTIVITY TEST
# =========================

async def test_connection():
    output = await chat_completion(
        model=TARGET_MODEL,
        system_prompt="You are a helpful assistant.",
        user_prompt="Reply with exactly: working",
        temperature=0.0,
        max_tokens=10,
        sleep_after=0.0,
    )
    print("Connectivity test output:", output)
    log(f"Connectivity test output: {output}")

await test_connection()


Connectivity test output: working

[2026-03-15 20:43:41] Connectivity test output: working



## Single-Example Evaluation

`evaluate_single` calls the target model on one navigation question, normalises both the predicted and gold answers, and returns a result dictionary including a binary `correct` flag.

In [22]:
# =========================
# SINGLE EXAMPLE EVALUATION
# =========================

async def evaluate_single(question: str, gold_answer: str, system_prompt: str) -> dict:
    user_prompt = f"""Question:
{question}

Respond with exactly one final answer: Yes or No.
"""

    raw_output = await chat_completion(
        model=TARGET_MODEL,
        system_prompt=system_prompt,
        user_prompt=user_prompt,
        temperature=TARGET_TEMPERATURE,
        max_tokens=256,
        sleep_after=SLEEP_BETWEEN_TARGET_CALLS,
    )

    pred = normalize_answer(raw_output)
    gold = normalize_answer(gold_answer)
    correct = int(pred == gold)

    return {
        "question": question,
        "gold_answer": gold_answer,
        "gold_norm": gold,
        "raw_output": raw_output,
        "pred_norm": pred,
        "correct": correct
    }


## Dataset Evaluation

`evaluate_prompt_on_df` loops over every row in a DataFrame, calls `evaluate_single` for each, and aggregates the results into an accuracy score and a list of per-example details. This is the core evaluator used by both the baseline run and the OPRO loop.

In [23]:
# =========================
# DATASET EVALUATION
# =========================

async def evaluate_prompt_on_df(system_prompt: str, df: pd.DataFrame, split_name: str = "train") -> dict:
    rows = []

    for _, row in df.iterrows():
        result = await evaluate_single(
            question=row["question"],
            gold_answer=row["answer"],
            system_prompt=system_prompt
        )
        rows.append(result)

    df_results = pd.DataFrame(rows)
    accuracy = df_results["correct"].mean()

    summary = {
        "split": split_name,
        "accuracy": float(accuracy),
        "n": int(len(df_results)),
        "num_correct": int(df_results["correct"].sum()),
        "details": df_results.to_dict(orient="records"),
    }
    return summary


## Baseline Evaluation

The naive `BASELINE_PROMPT` is evaluated on **both** the train and test sets before any optimisation. This establishes the starting accuracy that all subsequent improvements are measured against.

In [24]:
# BASELINE EVALUATION


async def run_baseline():
    log("Running baseline prompt on TRAIN...")
    train_baseline = await evaluate_prompt_on_df(BASELINE_PROMPT, df_train, split_name="train")

    log("Running baseline prompt on TEST...")
    test_baseline = await evaluate_prompt_on_df(BASELINE_PROMPT, df_test, split_name="test")

    log(f"Baseline train accuracy: {train_baseline['accuracy']:.3f}")
    log(f"Baseline test accuracy:  {test_baseline['accuracy']:.3f}")

    wandb.log({
        "baseline/train_accuracy": train_baseline["accuracy"],
        "baseline/test_accuracy": test_baseline["accuracy"],
    })

    return train_baseline, test_baseline

baseline_train, baseline_test = await run_baseline()


[2026-03-15 20:43:41] Running baseline prompt on TRAIN...
[2026-03-15 20:44:57] Running baseline prompt on TEST...
[2026-03-15 20:46:10] Baseline train accuracy: 0.600
[2026-03-15 20:46:10] Baseline test accuracy:  0.600


## Trajectory Formatter

Formats the list of previously evaluated prompts and their scores into a readable string for inclusion in the OPRO meta-prompt. Prompts are sorted **ascending** by accuracy so the best-scoring prompt appears last — closest to the model's generation point — and receives the most attention.

In [25]:
# =========================
# TRAJECTORY FORMATTER
# =========================

def format_trajectory(history: list[dict]) -> str:
    ordered = sorted(history, key=lambda x: x["train_accuracy"])
    blocks = []

    for i, item in enumerate(ordered, start=1):
        blocks.append(
            f"""Prompt {i}
Train Accuracy: {item['train_accuracy']:.3f}
Prompt:
{item['prompt']}
"""
        )

    return "\n" + ("\n" + "=" * 60 + "\n").join(blocks)


## Task Examples Builder

Samples a small number of `(question, correct answer)` pairs from the training set and formats them as a numbered list. These are embedded in the meta-prompt so the optimiser understands the exact format and difficulty of the task it is writing a prompt for.

In [26]:
# =========================
# TASK EXAMPLES FOR META-PROMPT
# =========================

def build_task_examples(df: pd.DataFrame, n_examples: int = 3) -> str:
    sampled = df.head(n_examples)
    pieces = []

    for i, row in enumerate(sampled.itertuples(index=False), start=1):
        pieces.append(
            f"""Example {i}
Question:
{row.question}

Correct answer:
{row.answer}
"""
        )

    return "\n" + ("\n" + "-" * 50 + "\n").join(pieces)


## OPRO Meta-Prompt

The meta-prompt is the instruction given to the **optimiser LLM**. It contains:

1. A description of the task and what the target model must do.
2. The full optimisation trajectory (past prompts ordered by score, lowest to highest).
3. A few concrete task examples so the optimiser understands the input/output format.
4. An explicit instruction to wrap the new prompt in `<PROMPT>...</PROMPT>` tags.

This is the core mechanism of OPRO: the optimiser is itself prompted to perform gradient-free optimisation.

In [27]:
# =========================
# OPRO META-PROMPT
# =========================

OPRO_META_PROMPT = """You are an expert prompt optimisation system.

Your job is to write a better SYSTEM PROMPT for a target language model that solves a spatial navigation reasoning task.

Task description:
The target model is given a text-based navigation question. It must determine whether, after following all instructions, the agent returns to the starting point.

The correct output should be either:
- Yes
- No

Below is the optimisation trajectory: previous system prompts and their training accuracies.
They are arranged in ascending order of accuracy.

{trajectory}

Below are a few example tasks from the dataset.

{task_examples}

Write ONE new system prompt that is likely to achieve a higher training accuracy than the previous prompts.

Requirements:
1. The new prompt must be meaningfully different from earlier prompts.
2. It should improve spatial reasoning reliability.
3. It should instruct the model to reason carefully but finish with exactly one final answer: Yes or No.
4. Do not include explanations outside the prompt.
5. Wrap the new prompt exactly inside these tags:

<PROMPT>
...
</PROMPT>
"""


## Optimiser Step

`propose_new_prompt` builds the meta-prompt from the current history, calls the optimiser LLM, and extracts the candidate prompt text. If no valid `<PROMPT>` tag is found it returns `None` and the loop skips that iteration.

In [28]:
# =========================
# OPTIMISER STEP
# =========================

async def propose_new_prompt(history: list[dict], df_examples: pd.DataFrame):
    trajectory = format_trajectory(history)
    task_examples = build_task_examples(df_examples, n_examples=3)

    user_prompt = OPRO_META_PROMPT.format(
        trajectory=trajectory,
        task_examples=task_examples
    )

    optimiser_response = await chat_completion(
        model=OPTIMISER_MODEL,
        system_prompt="You are a prompt optimiser. Return only the requested prompt wrapped in tags.",
        user_prompt=user_prompt,
        temperature=OPTIMISER_TEMPERATURE,
        max_tokens=900,
        sleep_after=SLEEP_BETWEEN_OPTIMISER_CALLS,
    )

    new_prompt = extract_prompt_from_optimiser_response(optimiser_response)
    return new_prompt, optimiser_response


## OPRO Loop

The main optimisation loop:

1. Evaluates all initial prompts on the training set to seed the history.
2. Iteratively proposes and evaluates new prompts using the optimiser.
3. Deduplicates proposals — if the optimiser repeats a prompt already in history it is skipped.
4. Saves the full trajectory to disk after each iteration.
5. Stops early if train accuracy reaches `TARGET_SCORE_STOP` or after `MAX_OPRO_ITERS` iterations.

In [29]:
# OPRO LOOP

async def run_opro(
    initial_prompts: list[str],
    df_train: pd.DataFrame,
    max_iters: int = 10,
    target_score_stop: float = 0.95
):
    history = []

    for idx, prompt in enumerate(initial_prompts, start=1):
        log(f"Evaluating initial prompt {idx}/{len(initial_prompts)} on TRAIN...")
        summary = await evaluate_prompt_on_df(prompt, df_train, split_name="train")

        history.append({
            "iteration": 0,
            "prompt": prompt,
            "train_accuracy": summary["accuracy"],
            "train_summary": summary
        })

        log(f"Initial prompt {idx} train accuracy: {summary['accuracy']:.3f}")

    best_so_far = max(history, key=lambda x: x["train_accuracy"])
    log(f"Initial best train accuracy: {best_so_far['train_accuracy']:.3f}")

    for it in range(1, max_iters + 1):
        log("=" * 80)
        log(f"Starting OPRO iteration {it}/{max_iters}")

        try:
            candidate_prompt, raw_optimiser_response = await propose_new_prompt(history, df_train)
        except Exception as e:
            log(f"Prompt proposal failed on iteration {it}: {type(e).__name__}: {str(e)}")
            continue

        log("Raw optimiser response:")
        log(raw_optimiser_response)

        log("Candidate prompt proposed by optimiser:")
        log(candidate_prompt)

        existing_prompts = {item["prompt"].strip() for item in history}
        if candidate_prompt.strip() in existing_prompts:
            log("Candidate prompt is a duplicate. Skipping evaluation.")
            continue

        summary = await evaluate_prompt_on_df(candidate_prompt, df_train, split_name="train")

        is_best = summary["accuracy"] > best_so_far["train_accuracy"]

        record = {
            "iteration": it,
            "prompt": candidate_prompt,
            "train_accuracy": summary["accuracy"],
            "train_summary": summary
        }
        history.append(record)

        wandb.log({
            "opro/iteration": it,
            "opro/train_accuracy": summary["accuracy"],
            "opro/best_train_accuracy": max(h["train_accuracy"] for h in history),
            "opro/prompt_word_count": len(candidate_prompt.split()),
            "opro/is_new_best": int(is_best),
        })

        if is_best:
            best_so_far = record
            log(f"New BEST prompt found at iteration {it}: {summary['accuracy']:.3f}")
        else:
            log(f"No improvement at iteration {it}. Candidate accuracy: {summary['accuracy']:.3f}")

        with open(TRAJECTORY_JSON, "w", encoding="utf-8") as f:
            json.dump(history, f, indent=2, ensure_ascii=False)

        if best_so_far["train_accuracy"] >= target_score_stop:
            log(f"Stopping early because target score {target_score_stop:.2f} was reached.")
            break

    return history, best_so_far


## Run OPRO

Kicks off the full optimisation run with the two initial prompts. After this cell completes, `best_prompt_record` holds the prompt that achieved the highest training accuracy.

In [30]:
# =========================
# RUN OPRO
# =========================

initial_prompt_pool = [
    BASELINE_PROMPT,
    SEED_PROMPT,
]

history, best_prompt_record = await run_opro(
    initial_prompts=initial_prompt_pool,
    df_train=df_train,
    max_iters=MAX_OPRO_ITERS,
    target_score_stop=TARGET_SCORE_STOP
)

best_prompt = best_prompt_record["prompt"]
best_train_acc = best_prompt_record["train_accuracy"]

print("\nBest train accuracy:", best_train_acc)
print("\nBest prompt:\n")
print(best_prompt)


[2026-03-15 20:46:11] Evaluating initial prompt 1/2 on TRAIN...
[2026-03-15 20:47:25] Initial prompt 1 train accuracy: 0.600
[2026-03-15 20:47:25] Evaluating initial prompt 2/2 on TRAIN...
[2026-03-15 20:48:42] Initial prompt 2 train accuracy: 0.550
[2026-03-15 20:48:42] Initial best train accuracy: 0.600
[2026-03-15 20:48:42] ================================================================================
[2026-03-15 20:48:42] Starting OPRO iteration 1/10
[2026-03-15 20:48:52] Raw optimiser response:
[2026-03-15 20:48:52] <PROMPT>
You are a spatial reasoning expert. Your task is to determine if a sequence of movements returns to the origin (0, 0).

Maintain an internal x and y coordinate, initialized to (0, 0).
Process each instruction sequentially, updating x and y accordingly:
- "forward": y + 1
- "backward": y - 1
- "left": x - 1
- "right": x + 1
- "turn right": Rotate facing direction clockwise (does not change coordinates).
- "turn left": Rotate facing direction counter-clockwise

## Final Generalisation Check

The best prompt found during training is evaluated on the **held-out test set exactly once**. Keeping the test set sealed until this point prevents any information from test performance influencing which prompt is selected.

In [31]:
# =========================
# FINAL GENERALISATION CHECK
# =========================

log("Evaluating BEST prompt on held-out TEST set...")
final_test_summary = await evaluate_prompt_on_df(best_prompt, df_test, split_name="test")

log(f"Final best prompt TEST accuracy: {final_test_summary['accuracy']:.3f}")

wandb.log({
    "final/best_train_accuracy": best_train_acc,
    "final/best_test_accuracy": final_test_summary["accuracy"],
    "final/train_uplift": best_train_acc - baseline_train["accuracy"],
    "final/test_uplift": final_test_summary["accuracy"] - baseline_test["accuracy"],
})
wandb_run_url = wandb.run.url
wandb.finish()
log("W&B run finished.")
from IPython.display import display, HTML
display(HTML(f'<b>W&amp;B Run (finished):</b> <a href="{wandb_run_url}" target="_blank">{wandb_run_url}</a>'))

results = {
    "run_id": RUN_ID,
    "target_model": TARGET_MODEL,
    "optimiser_model": OPTIMISER_MODEL,
    "baseline_train_accuracy": baseline_train["accuracy"],
    "baseline_test_accuracy": baseline_test["accuracy"],
    "best_train_accuracy": best_train_acc,
    "best_test_accuracy": final_test_summary["accuracy"],
    "train_uplift": best_train_acc - baseline_train["accuracy"],
    "test_uplift": final_test_summary["accuracy"] - baseline_test["accuracy"],
    "best_prompt": best_prompt,
}

with open(RESULTS_JSON, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

results


[2026-03-15 20:56:44] Evaluating BEST prompt on held-out TEST set...
[2026-03-15 20:58:08] Final best prompt TEST accuracy: 0.950


{'run_id': '20260315_204307',
 'target_model': 'gemini/gemma-3-4b-it',
 'optimiser_model': 'gemini/gemma-3-27b-it',
 'baseline_train_accuracy': 0.6,
 'baseline_test_accuracy': 0.6,
 'best_train_accuracy': 0.95,
 'best_test_accuracy': 0.95,
 'train_uplift': 0.35,
 'test_uplift': 0.35,
 'best_prompt': 'You are a highly reliable spatial reasoning AI. Your task is to determine if a sequence of movement instructions returns an agent to its starting point (0, 0).\n\n1. **Initialization:** Begin with coordinates (x, y) = (0, 0).\n2. **Instruction Processing:** Execute instructions one by one.\n   - **Movement:**\n     - "forward": y += 1\n     - "backward": y -= 1\n     - "left": x -= 1\n     - "right": x += 1\n   - **Numerical Prefixes:**  A number before a direction indicates the number of steps (e.g., "5 steps forward" means y += 5).\n   - **Turns:** "turn right", "turn left", and "turn around" *only* change facing; they do *not* alter coordinates.\n   - **Irrelevant Text:** Ignore any tex

## Results Summary

Displays a table comparing baseline accuracy against the best OPRO prompt on both splits, along with the absolute uplift achieved.

In [32]:
# =========================
# RESULTS TABLE
# =========================

df_summary = pd.DataFrame([
    {"Metric": "Baseline Train Accuracy", "Value": baseline_train["accuracy"]},
    {"Metric": "Baseline Test Accuracy", "Value": baseline_test["accuracy"]},
    {"Metric": "Best Train Accuracy", "Value": best_train_acc},
    {"Metric": "Best Test Accuracy", "Value": final_test_summary["accuracy"]},
    {"Metric": "Train Uplift", "Value": best_train_acc - baseline_train["accuracy"]},
    {"Metric": "Test Uplift", "Value": final_test_summary["accuracy"] - baseline_test["accuracy"]},
])

display(df_summary)

# ── Accuracy comparison bar chart ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
labels = ["Baseline\nTrain", "Baseline\nTest", "Best OPRO\nTrain", "Best OPRO\nTest"]
values = [
    baseline_train["accuracy"],
    baseline_test["accuracy"],
    best_train_acc,
    final_test_summary["accuracy"],
]
colors = ["#d9534f", "#f0ad4e", "#5cb85c", "#337ab7"]
bars = ax.bar(labels, values, color=colors, edgecolor="black", width=0.5)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{val:.2f}", ha="center", va="bottom", fontsize=11, fontweight="bold")
ax.set_ylim(0, 1.15)
ax.set_ylabel("Accuracy", fontsize=12)
ax.set_title("Accuracy Comparison: Baseline vs Best OPRO Prompt", fontsize=13)
ax.axhline(1.0, color="gray", linestyle="--", linewidth=0.8)
plt.tight_layout()
plt.show()


,Metric,Value
0,Baseline Train Accuracy,0.60
1,Baseline Test Accuracy,0.60
2,Best Train Accuracy,0.95
3,Best Test Accuracy,0.95
4,Train Uplift,0.35
5,Test Uplift,0.35


## Trajectory Inspection

Shows every prompt evaluated during the OPRO loop, sorted by training accuracy, with a short preview of each prompt. Makes it easy to see which iterations produced the biggest jumps.

In [33]:
# =========================
# TRAJECTORY INSPECTION
# =========================

trajectory_df = pd.DataFrame([
    {
        "iteration": item["iteration"],
        "train_accuracy": item["train_accuracy"],
        "prompt_preview": item["prompt"][:200].replace("\n", " ") + ("..." if len(item["prompt"]) > 200 else "")
    }
    for item in history
]).sort_values(["train_accuracy", "iteration"], ascending=[False, True])

display(trajectory_df)


,iteration,train_accuracy,prompt_preview
7,6,0.95,You are a highly reliable spatial reasoning AI...
2,1,0.65,You are a spatial reasoning expert. Your task ...
3,2,0.65,You are a highly accurate spatial navigator. Y...
4,3,0.65,You are a meticulous spatial reasoner tasked w...
0,0,0.60,Solve the navigation problem. Determine whethe...
5,4,0.60,You are a spatial reasoning engine. Determine ...
1,0,0.55,You are solving a spatial navigation reasoning...
6,5,0.55,You are a world-class spatial reasoning AI. Yo...


## Trajectory Visualisation
Plots training accuracy for every OPRO candidate, with a running-best overlay and a baseline reference line.

In [ ]:
# =========================
# TRAJECTORY VISUALISATION
# =========================

iters      = [item["iteration"] for item in history]
accs       = [item["train_accuracy"] for item in history]
running_best = [max(accs[:i + 1]) for i in range(len(accs))]

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(iters, accs, marker="o", linewidth=1.5, color="steelblue",
        label="Candidate train accuracy", zorder=3)
ax.plot(iters, running_best, linestyle="--", linewidth=2, color="darkorange",
        label="Running best train accuracy", zorder=2)

best_iter_idx = accs.index(max(accs))
ax.axvline(x=iters[best_iter_idx], color="green", linestyle=":",
           linewidth=1.5, label=f"Best iteration ({iters[best_iter_idx]})")

ax.axhline(y=baseline_train["accuracy"], color="red", linestyle="-.",
           linewidth=1.2, label=f'Baseline train ({baseline_train["accuracy"]:.2f})')

ax.set_xlabel("OPRO Iteration", fontsize=12)
ax.set_ylabel("Train Accuracy", fontsize=12)
ax.set_title("OPRO Optimisation Trajectory — BBH Navigate", fontsize=14)
ax.set_ylim(-0.05, 1.05)
ax.set_xticks(iters)
ax.legend(fontsize=10)
ax.grid(axis="y", alpha=0.3)
PLOT_FILE = OUTPUT_DIR / "trajectory_plot.png"
plt.tight_layout()
plt.savefig(PLOT_FILE, dpi=150)
plt.show()
log(f"Trajectory plot saved to: {PLOT_FILE.resolve()}")


## Save Best Prompt

Writes the best prompt text to `best_prompt.txt` so it is available as a standalone file in the submission bundle.

In [34]:
# =========================
# SAVE BEST PROMPT
# =========================

BEST_PROMPT_FILE = OUTPUT_DIR / "best_prompt.txt"

with open(BEST_PROMPT_FILE, "w", encoding="utf-8") as f:
    f.write(best_prompt)

print("Saved best prompt to:", BEST_PROMPT_FILE.resolve())


Saved best prompt to: /content/apo_run_20260315_204307/best_prompt.txt


## Error Analysis *(Optional)*

Compares the baseline and best-prompt predictions on the test set side by side. Highlights which individual examples improved, which regressed, and which remained incorrect — useful for diagnosing where the optimised prompt still struggles.

In [35]:
# =========================
# OPTIONAL ERROR ANALYSIS
# =========================

baseline_test_df = pd.DataFrame(baseline_test["details"])
final_test_df = pd.DataFrame(final_test_summary["details"])

comparison_df = baseline_test_df[["question", "gold_norm", "pred_norm", "correct"]].rename(
    columns={"pred_norm": "baseline_pred", "correct": "baseline_correct"}
).merge(
    final_test_df[["question", "pred_norm", "correct"]].rename(
        columns={"pred_norm": "best_pred", "correct": "best_correct"}
    ),
    on="question",
    how="inner"
)

improved_cases = comparison_df[
    (comparison_df["baseline_correct"] == 0) & (comparison_df["best_correct"] == 1)
]

regressed_cases = comparison_df[
    (comparison_df["baseline_correct"] == 1) & (comparison_df["best_correct"] == 0)
]

print("Improved cases:", len(improved_cases))
print("Regressed cases:", len(regressed_cases))

display(improved_cases.head(10))
display(regressed_cases.head(10))


Improved cases: 7
Regressed cases: 0


,question,gold_norm,baseline_pred,baseline_correct,best_pred,best_correct
0,"If you follow these instructions, do you retur...",no,yes,0,no,1
1,"If you follow these instructions, do you retur...",yes,no,0,yes,1
6,"If you follow these instructions, do you retur...",yes,no,0,yes,1
12,"If you follow these instructions, do you retur...",yes,no,0,yes,1
16,"If you follow these instructions, do you retur...",yes,no,0,yes,1
17,"If you follow these instructions, do you retur...",yes,no,0,yes,1
18,"If you follow these instructions, do you retur...",yes,no,0,yes,1


,question,gold_norm,baseline_pred,baseline_correct,best_pred,best_correct


## Few-Shot Exemplar Search — Helpers *(Optional Improvement)*

Helper functions for the few-shot extension:

- `build_few_shot_prompt` appends solved examples to a base prompt.
- `objective_with_length_penalty` computes a penalised score: `accuracy − 0.002 × (words / 100)`, discouraging unnecessarily long prompts.
- `get_correct_train_examples` collects the training examples the best OPRO prompt already solves correctly.
- `choose_balanced_candidate_exemplars` selects up to 2 Yes-class and 2 No-class examples to avoid class imbalance.

In [36]:
# =========================
# OPTIONAL IMPROVEMENT:
# FEW-SHOT EXEMPLAR SEARCH HELPERS
# ADD THIS AFTER YOUR ERROR ANALYSIS CELL
# =========================

import itertools

def build_few_shot_prompt(base_prompt: str, exemplars: list[dict]) -> str:
    blocks = []
    for i, ex in enumerate(exemplars, start=1):
        blocks.append(
            f"""Example {i}
Question:
{ex["question"]}

Correct final answer:
{ex["answer"]}"""
        )

    examples_text = "\n\n".join(blocks)

    return f"""{base_prompt}

Use the following solved examples to understand the expected reasoning pattern and output format.

{examples_text}

For the real task:
- simulate carefully
- track movement and facing direction exactly
- verify whether the final position is the starting position
- answer with exactly one word: Yes or No
"""


def approx_prompt_word_count(prompt: str) -> int:
    return len(prompt.split())


def objective_with_length_penalty(
    accuracy: float,
    prompt: str,
    penalty_per_100_words: float = 0.002
) -> float:
    words = approx_prompt_word_count(prompt)
    penalty = penalty_per_100_words * (words / 100.0)
    return accuracy - penalty


async def get_correct_train_examples(system_prompt: str, df: pd.DataFrame) -> list[dict]:
    summary = await evaluate_prompt_on_df(system_prompt, df, split_name="train_for_exemplar_search")
    correct_examples = []

    for row in summary["details"]:
        if row["correct"] == 1:
            correct_examples.append({
                "question": row["question"],
                "answer": row["gold_answer"],
                "pred_norm": row["pred_norm"]
            })

    return correct_examples


def choose_balanced_candidate_exemplars(correct_examples: list[dict], max_yes: int = 2, max_no: int = 2) -> list[dict]:
    yes_examples = []
    no_examples = []

    for ex in correct_examples:
        ans = normalize_answer(ex["answer"])
        if ans == "yes" and len(yes_examples) < max_yes:
            yes_examples.append(ex)
        elif ans == "no" and len(no_examples) < max_no:
            no_examples.append(ex)

    return yes_examples + no_examples


## Few-Shot Variant Search *(Optional Improvement)*

Exhaustively evaluates every size-2 combination of candidate exemplars on the training set only. Each variant is scored with the length-penalised objective and results are sorted best-first.

In [37]:
# =========================
# OPTIONAL IMPROVEMENT:
# SEARCH FEW-SHOT VARIANTS ON TRAIN ONLY
# ADD THIS AFTER THE PREVIOUS NEW CELL
# =========================

async def search_few_shot_variants(
    base_prompt: str,
    df_train: pd.DataFrame,
    candidate_exemplars: list[dict],
    combo_sizes=(2,),
    penalty_per_100_words: float = 0.002
):
    results = []

    for k in combo_sizes:
        for combo in itertools.combinations(candidate_exemplars, k):
            candidate_prompt = build_few_shot_prompt(base_prompt, list(combo))

            summary = await evaluate_prompt_on_df(
                candidate_prompt,
                df_train,
                split_name=f"train_fewshot_k{k}"
            )

            acc = summary["accuracy"]
            obj = objective_with_length_penalty(
                accuracy=acc,
                prompt=candidate_prompt,
                penalty_per_100_words=penalty_per_100_words
            )

            results.append({
                "num_exemplars": k,
                "train_accuracy": acc,
                "objective": obj,
                "prompt_words": approx_prompt_word_count(candidate_prompt),
                "prompt": candidate_prompt,
                "summary": summary,
                "examples": list(combo),
            })

            log(
                f"Few-shot candidate evaluated | "
                f"k={k} | train_acc={acc:.3f} | objective={obj:.3f} | words={approx_prompt_word_count(candidate_prompt)}"
            )

    results = sorted(results, key=lambda x: (x["objective"], x["train_accuracy"]), reverse=True)
    return results


correct_train_examples = await get_correct_train_examples(best_prompt, df_train)
candidate_exemplars = choose_balanced_candidate_exemplars(correct_train_examples, max_yes=2, max_no=2)

print("Correct train examples under current best prompt:", len(correct_train_examples))
print("Candidate exemplar pool size:", len(candidate_exemplars))

few_shot_results = await search_few_shot_variants(
    base_prompt=best_prompt,
    df_train=df_train,
    candidate_exemplars=candidate_exemplars,
    combo_sizes=(2,),
    penalty_per_100_words=0.002
)

few_shot_df = pd.DataFrame([
    {
        "rank": i + 1,
        "num_exemplars": r["num_exemplars"],
        "train_accuracy": r["train_accuracy"],
        "objective": r["objective"],
        "prompt_words": r["prompt_words"]
    }
    for i, r in enumerate(few_shot_results)
])

display(few_shot_df.head(10))


Correct train examples under current best prompt: 19
Candidate exemplar pool size: 4
[2026-03-15 21:00:55] Few-shot candidate evaluated | k=2 | train_acc=0.950 | objective=0.944 | words=296
[2026-03-15 21:02:16] Few-shot candidate evaluated | k=2 | train_acc=0.950 | objective=0.944 | words=319
[2026-03-15 21:03:41] Few-shot candidate evaluated | k=2 | train_acc=0.950 | objective=0.944 | words=307
[2026-03-15 21:05:02] Few-shot candidate evaluated | k=2 | train_acc=0.950 | objective=0.944 | words=312
[2026-03-15 21:06:22] Few-shot candidate evaluated | k=2 | train_acc=0.900 | objective=0.894 | words=300
[2026-03-15 21:07:44] Few-shot candidate evaluated | k=2 | train_acc=0.900 | objective=0.894 | words=323


,rank,num_exemplars,train_accuracy,objective,prompt_words
0,1,2,0.95,0.94408,296
1,2,2,0.95,0.94386,307
2,3,2,0.95,0.94376,312
3,4,2,0.95,0.94362,319
4,5,2,0.90,0.89400,300
5,6,2,0.90,0.89354,323


## Select Final Prompt *(Optional Improvement)*

Compares the best OPRO prompt against the best few-shot variant using the penalised objective. Whichever scores higher becomes `selected_final_prompt` — the prompt evaluated on the test set.

In [38]:
# =========================
# OPTIONAL IMPROVEMENT:
# SELECT FINAL PROMPT
# ADD THIS AFTER THE PREVIOUS NEW CELL
# =========================

few_shot_best = few_shot_results[0] if len(few_shot_results) > 0 else None

current_best_objective = objective_with_length_penalty(
    accuracy=best_train_acc,
    prompt=best_prompt,
    penalty_per_100_words=0.002
)

print("Current best OPRO train accuracy:", best_train_acc)
print("Current best OPRO objective:", round(current_best_objective, 4))

if few_shot_best is not None:
    print("Best few-shot train accuracy:", few_shot_best["train_accuracy"])
    print("Best few-shot objective:", round(few_shot_best["objective"], 4))
    print("Best few-shot prompt words:", few_shot_best["prompt_words"])

if few_shot_best is not None and few_shot_best["objective"] > current_best_objective:
    selected_final_prompt = few_shot_best["prompt"]
    selected_final_train_acc = few_shot_best["train_accuracy"]
    selected_prompt_source = "few_shot_variant"
    print("\nSelected final prompt = FEW-SHOT VARIANT")
else:
    selected_final_prompt = best_prompt
    selected_final_train_acc = best_train_acc
    selected_prompt_source = "original_best_opro_prompt"
    print("\nSelected final prompt = ORIGINAL BEST OPRO PROMPT")


Current best OPRO train accuracy: 0.95
Current best OPRO objective: 0.9464
Best few-shot train accuracy: 0.95
Best few-shot objective: 0.9441
Best few-shot prompt words: 296

Selected final prompt = ORIGINAL BEST OPRO PROMPT


## Final Test Evaluation — Selected Prompt *(Optional Improvement)*

Evaluates the selected final prompt on the held-out test set and saves the results to disk. The test set is touched only here, preserving the integrity of the evaluation.

In [39]:
# =========================
# OPTIONAL IMPROVEMENT:
# FINAL TEST EVALUATION OF SELECTED FINAL PROMPT
# ADD THIS AFTER THE PREVIOUS NEW CELL
# =========================

log(f"Evaluating selected final prompt on held-out TEST | source={selected_prompt_source}")

selected_final_test_summary = await evaluate_prompt_on_df(
    selected_final_prompt,
    df_test,
    split_name="test_selected_final"
)

SELECTED_RESULTS_JSON = OUTPUT_DIR / "selected_final_results.json"
SELECTED_PROMPT_FILE = OUTPUT_DIR / "selected_final_prompt.txt"

selected_final_results = {
    "run_id": RUN_ID,
    "target_model": TARGET_MODEL,
    "optimiser_model": OPTIMISER_MODEL,
    "selected_prompt_source": selected_prompt_source,
    "baseline_train_accuracy": baseline_train["accuracy"],
    "baseline_test_accuracy": baseline_test["accuracy"],
    "best_opro_train_accuracy": best_train_acc,
    "best_opro_test_accuracy": final_test_summary["accuracy"],
    "selected_final_train_accuracy": selected_final_train_acc,
    "selected_final_test_accuracy": selected_final_test_summary["accuracy"],
    "train_uplift_vs_baseline": selected_final_train_acc - baseline_train["accuracy"],
    "test_uplift_vs_baseline": selected_final_test_summary["accuracy"] - baseline_test["accuracy"],
    "selected_final_prompt": selected_final_prompt,
}

with open(SELECTED_RESULTS_JSON, "w", encoding="utf-8") as f:
    json.dump(selected_final_results, f, indent=2, ensure_ascii=False)

with open(SELECTED_PROMPT_FILE, "w", encoding="utf-8") as f:
    f.write(selected_final_prompt)

selected_final_results


[2026-03-15 21:07:44] Evaluating selected final prompt on held-out TEST | source=original_best_opro_prompt


{'run_id': '20260315_204307',
 'target_model': 'gemini/gemma-3-4b-it',
 'optimiser_model': 'gemini/gemma-3-27b-it',
 'selected_prompt_source': 'original_best_opro_prompt',
 'baseline_train_accuracy': 0.6,
 'baseline_test_accuracy': 0.6,
 'best_opro_train_accuracy': 0.95,
 'best_opro_test_accuracy': 0.95,
 'selected_final_train_accuracy': 0.95,
 'selected_final_test_accuracy': 0.85,
 'train_uplift_vs_baseline': 0.35,
 'test_uplift_vs_baseline': 0.25,
 'selected_final_prompt': 'You are a highly reliable spatial reasoning AI. Your task is to determine if a sequence of movement instructions returns an agent to its starting point (0, 0).\n\n1. **Initialization:** Begin with coordinates (x, y) = (0, 0).\n2. **Instruction Processing:** Execute instructions one by one.\n   - **Movement:**\n     - "forward": y += 1\n     - "backward": y -= 1\n     - "left": x -= 1\n     - "right": x += 1\n   - **Numerical Prefixes:**  A number before a direction indicates the number of steps (e.g., "5 steps for

## Final Comparison Table *(Optional Improvement)*

Side-by-side accuracy comparison of all three versions:

- Baseline (naive prompt)
- Best OPRO prompt
- Selected final prompt (best of OPRO vs few-shot)

In [40]:
# =========================
# OPTIONAL IMPROVEMENT:
# FINAL COMPARISON TABLE
# ADD THIS AFTER THE PREVIOUS NEW CELL
# =========================

final_comparison_df = pd.DataFrame([
    {
        "Version": "Baseline",
        "Train Accuracy": baseline_train["accuracy"],
        "Test Accuracy": baseline_test["accuracy"],
        "Prompt Source": "naive_baseline"
    },
    {
        "Version": "Best OPRO Prompt",
        "Train Accuracy": best_train_acc,
        "Test Accuracy": final_test_summary["accuracy"],
        "Prompt Source": "opro"
    },
    {
        "Version": "Selected Final Prompt",
        "Train Accuracy": selected_final_train_acc,
        "Test Accuracy": selected_final_test_summary["accuracy"],
        "Prompt Source": selected_prompt_source
    }
])

display(final_comparison_df)


,Version,Train Accuracy,Test Accuracy,Prompt Source
0,Baseline,0.60,0.60,naive_baseline
1,Best OPRO Prompt,0.95,0.95,opro
2,Selected Final Prompt,0.95,0.85,original_best_opro_prompt


## Writeup

Generates `writeup.md` and saves it to the output directory for inclusion in the submission zip.

In [41]:
# =========================
# WRITEUP FILE
# REPLACE YOUR CURRENT WRITEUP CELL WITH THIS
# =========================

WRITEUP_FILE = OUTPUT_DIR / "writeup.md"

writeup_text = f"""# Assignment 2 Writeup: Automated Prompt Optimisation with OPRO on BBH Navigate

## Objective
The goal of this assignment was to build a complete Automated Prompt Optimisation (APO) pipeline using the OPRO strategy for the BIG-Bench Hard (BBH) Navigate task.

## Experimental Setup
- Target model: {TARGET_MODEL}
- Optimiser model: {OPTIMISER_MODEL}
- Dataset: `lukaemon/bbh`, configuration `navigate`
- Training split: first 20 records `[0:20]`
- Test split: next 20 records `[20:40]`

## Baseline
I first established a naive baseline on both the training and test sets before any optimisation.

- Baseline train accuracy: {baseline_train["accuracy"]:.3f}
- Baseline test accuracy: {baseline_test["accuracy"]:.3f}

## OPRO Method
I implemented an automated iterative optimisation loop:
1. Evaluate candidate prompts on the training set
2. Store prompt-score history
3. Build a meta-prompt containing the optimisation trajectory
4. Ask the optimiser LLM to propose a better prompt
5. Extract the prompt automatically
6. Evaluate it on the training set
7. Repeat until a stopping condition is reached
8. Evaluate the best prompt on the held-out test set

## Core OPRO Results
- Best OPRO train accuracy: {best_train_acc:.3f}
- Best OPRO test accuracy: {final_test_summary["accuracy"]:.3f}
- Train uplift vs baseline: {best_train_acc - baseline_train["accuracy"]:.3f}
- Test uplift vs baseline: {final_test_summary["accuracy"] - baseline_test["accuracy"]:.3f}

## Optional Challenge Extension
As an additional improvement, I implemented a small train-only few-shot exemplar search.
Correctly solved training examples were used as candidate exemplars, and compact few-shot prompt variants were compared using:
- training accuracy
- a mild prompt-length penalty to discourage unnecessarily long prompts

This extension preserved the required train/test separation because the test set remained untouched until the final evaluation step.

## Selected Final Prompt Results
- Selected final prompt source: {selected_prompt_source}
- Selected final train accuracy: {selected_final_train_acc:.3f}
- Selected final test accuracy: {selected_final_test_summary["accuracy"]:.3f}
- Train uplift vs baseline: {selected_final_train_acc - baseline_train["accuracy"]:.3f}
- Test uplift vs baseline: {selected_final_test_summary["accuracy"] - baseline_test["accuracy"]:.3f}

## Observations
The strongest prompts shared several useful properties:
- explicit initialisation at coordinate `(0, 0)`
- explicit tracking of facing direction
- step-by-step accumulation of movement
- clear handling of turns and numerical prefixes
- a strict final output format of `Yes` or `No`

The final train/test gap remained small, suggesting reasonable generalisation rather than severe overfitting.

## Conclusion
The final system achieved a substantial improvement over the naive baseline. The results suggest that automated prompt optimisation can materially improve reasoning reliability on the Navigate task, and that the learned prompt generalises well to unseen examples.
"""

with open(WRITEUP_FILE, "w", encoding="utf-8") as f:
    f.write(writeup_text)

print("Updated writeup saved to:", WRITEUP_FILE.resolve())
print(writeup_text[:1500])


Updated writeup saved to: /content/apo_run_20260315_204307/writeup.md
# Assignment 2 Writeup: Automated Prompt Optimisation with OPRO on BBH Navigate

## Objective
The goal of this assignment was to build a complete Automated Prompt Optimisation (APO) pipeline using the OPRO strategy for the BIG-Bench Hard (BBH) Navigate task.

## Experimental Setup
- Target model: gemini/gemma-3-4b-it
- Optimiser model: gemini/gemma-3-27b-it
- Dataset: `lukaemon/bbh`, configuration `navigate`
- Training split: first 20 records `[0:20]`
- Test split: next 20 records `[20:40]`

## Baseline
I first established a naive baseline on both the training and test sets before any optimisation.

- Baseline train accuracy: 0.600
- Baseline test accuracy: 0.600

## OPRO Method
I implemented an automated iterative optimisation loop:
1. Evaluate candidate prompts on the training set
2. Store prompt-score history
3. Build a meta-prompt containing the optimisation trajectory
4. Ask the optimiser LLM to propose a better

## AI Contribution File

Generates `AI_CONTRIBUTION.md` documenting how AI tools were used to assist with this assignment, including verbatim prompt and response excerpts as required.

In [42]:
# =========================
# AI CONTRIBUTION FILE
# REPLACE YOUR CURRENT AI CONTRIBUTION CELL WITH THIS
# =========================

AI_CONTRIBUTION_FILE = OUTPUT_DIR / "AI_CONTRIBUTION.md"

ai_contribution_text = f"""# AI_CONTRIBUTION.md

## AI Tools Used
- ChatGPT

## How AI Was Used
I used ChatGPT as a coding and writing assistant to help me:
1. structure the notebook into clearly separated sections
2. draft Python helper functions for LiteLLM calls, evaluation, parsing, logging, and file export
3. draft the OPRO meta-prompt and optimisation loop structure
4. draft the explanatory writeup and submission packaging logic

## Example Prompt(s) Given to AI
1. "Please provide notebook code blocks that implement the APO assignment for BBH Navigate using LiteLLM, Gemini Gemma target and optimiser models, train/test split [0:20] and [20:40], baseline evaluation, OPRO loop, prompt extraction, logging, and final held-out test evaluation."

2. "Please provide additional code blocks that improve the model further while preserving the assignment rules, especially train-only optimisation and held-out test evaluation."

## Verbatim AI Response Excerpt
The AI suggested code for:
- dataset loading with the exact required split
- baseline train/test evaluation
- an iterative OPRO loop using prompt-score history
- automatic extraction of improved prompts
- optional few-shot exemplar selection on the training set only
- output logging and submission files

## My Role
I reviewed, edited, executed, and validated all generated code myself.
I selected the final design, checked that the implementation matched the assignment brief, and interpreted the final results.

## Final Responsibility
All submitted code and written analysis were reviewed by me before submission.
"""

with open(AI_CONTRIBUTION_FILE, "w", encoding="utf-8") as f:
    f.write(ai_contribution_text)

print("Updated AI contribution file saved to:", AI_CONTRIBUTION_FILE.resolve())
print(ai_contribution_text[:1500])


Updated AI contribution file saved to: /content/apo_run_20260315_204307/AI_CONTRIBUTION.md
# AI_CONTRIBUTION.md

## AI Tools Used
- ChatGPT

## How AI Was Used
I used ChatGPT as a coding and writing assistant to help me:
1. structure the notebook into clearly separated sections
2. draft Python helper functions for LiteLLM calls, evaluation, parsing, logging, and file export
3. draft the OPRO meta-prompt and optimisation loop structure
4. draft the explanatory writeup and submission packaging logic

## Example Prompt(s) Given to AI
1. "Please provide notebook code blocks that implement the APO assignment for BBH Navigate using LiteLLM, Gemini Gemma target and optimiser models, train/test split [0:20] and [20:40], baseline evaluation, OPRO loop, prompt extraction, logging, and final held-out test evaluation."

2. "Please provide additional code blocks that improve the model further while preserving the assignment rules, especially train-only optimisation and held-out test evaluation."

#

## Show Output Files

Lists every file created in the run output directory.

In [43]:
# =========================
# SHOW CREATED FILES
# =========================

print("Files created in:", OUTPUT_DIR.resolve())
for p in sorted(OUTPUT_DIR.iterdir()):
    print("-", p.name)


Files created in: /content/apo_run_20260315_204307
- AI_CONTRIBUTION.md
- best_prompt.txt
- final_results.json
- opro_log.txt
- selected_final_prompt.txt
- selected_final_results.json
- trajectory.json
- writeup.md


## Submission Bundle

Packages all deliverables into a single `.zip` file ready for submission: log file, trajectory JSON, results JSON, best prompt, writeup, and AI contribution file.

In [44]:
# =========================
# ZIP DELIVERABLES
# REPLACE YOUR CURRENT ZIP CELL WITH THIS
# =========================

import zipfile

FINAL_ZIP_PATH = OUTPUT_DIR / "assignment_submission_bundle_FINAL.zip"

files_to_include = [
    LOG_FILE,
    TRAJECTORY_JSON,
    RESULTS_JSON,
    BEST_PROMPT_FILE,
    WRITEUP_FILE,
    AI_CONTRIBUTION_FILE
]

# Add optional improved files if they exist
optional_files = [
    OUTPUT_DIR / "selected_final_results.json",
    OUTPUT_DIR / "selected_final_prompt.txt"
]

for p in optional_files:
    if p.exists():
        files_to_include.append(p)

with zipfile.ZipFile(FINAL_ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    for p in files_to_include:
        if p.exists():
            z.write(p, arcname=p.name)

print("Final ZIP created:", FINAL_ZIP_PATH.resolve())
print("Included files:")
for p in files_to_include:
    if p.exists():
        print("-", p.name)


Final ZIP created: /content/apo_run_20260315_204307/assignment_submission_bundle_FINAL.zip
Included files:
- opro_log.txt
- trajectory.json
- final_results.json
- best_prompt.txt
- writeup.md
- AI_CONTRIBUTION.md
- selected_final_results.json
- selected_final_prompt.txt


## Fallback Models *(Optional)*

If the default Gemma models are unavailable or hitting rate limits, switch to the standard Gemini 1.5 Flash / Pro models here and re-run from the connectivity test cell.

In [45]:
TARGET_MODEL = "gemini/gemini-1.5-flash"
OPTIMISER_MODEL = "gemini/gemini-1.5-pro"

print("Switched models to fallback Gemini models.")
print("Target model:", TARGET_MODEL)
print("Optimiser model:", OPTIMISER_MODEL)


Switched models to fallback Gemini models.
Target model: gemini/gemini-1.5-flash
Optimiser model: gemini/gemini-1.5-pro
